# Task 4: Final Semantic Embeddings and Quality Diagnostics

This notebook builds semantic embeddings for the final preambles dataset (`data/processed/preamble_final.csv`) using the `all-mpnet-base-v2` model:
- Filters for clusterable preambles (`status` in `['preamble_ok', 'preamble_cut']`), verifying exactly 158 rows.
- Preserves the file order and saves `data/processed/final_country_order.csv`.
- Generates two text variants:
  - **Variant A ("raw")**: `extracted_text` with whitespace collapsed to single spaces.
  - **Variant B ("clean")**: `extracted_text` cleaned using the exact preprocessing and stopword list from `01_data_preprocessing.ipynb` cell 44.
- Tokenizes and chunks texts exceeding 300 tokens:
  - **Variant A**: Greedy sentence packing at sentence boundaries (`.?!`), splitting oversized sentences by tokens.
  - **Variant B**: Consecutive windows of at most 300 tokens.
- Generates chunk embeddings, pools chunks per country via token-weighted mean, and L2-normalises to unit length.
- Saves final embeddings to `data/processed/emb_final_raw.npy` and `data/processed/emb_final_clean.npy`.
- Conducts seven quality and integrity checks: shape/NaN/norm assertions, determinism verification, chunk distribution analysis, short preamble identification, worked chunking example, pairwise cosine similarities, and Spearman length correlation diagnostics.

## 1. Setup and Environment
Import required libraries, verify directory paths, configure Hugging Face offline cache, and initialize the `all-mpnet-base-v2` embedding model.

In [1]:
import os
import re
import json
import html
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from scipy.stats import spearmanr
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

data_dir = '../data/processed'

# Initialize model and tokenizer
model = SentenceTransformer('all-mpnet-base-v2')
tokenizer = model.tokenizer
print("Model and tokenizer successfully initialized:")
print(model)


/home/devpanchal/Projects/PreambleMap/venv/lib64/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model and tokenizer successfully initialized:
SentenceTransformer(
  (0): Transformer({'max_seq_length': 384, 'do_lower_case': False, 'architecture': 'MPNetModel'})
  (1): Pooling({'word_embedding_dimension': 768, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)


## 2. Load Final Dataset and Lock Country Order
Load `data/processed/preamble_final.csv`, filter for clusterable countries with `status` in `('preamble_ok', 'preamble_cut')`, assert exactly 158 rows, print the country list in that order, and save `data/processed/final_country_order.csv`.

In [2]:
final_csv_path = os.path.join(data_dir, 'preamble_final.csv')
df_full = pd.read_csv(final_csv_path)
print(f"Total rows in {final_csv_path}: {len(df_full)}")

# Filter for clusterable preambles
df = df_full[df_full['status'].isin(['preamble_ok', 'preamble_cut'])].copy().reset_index(drop=True)
print(f"Clusterable rows (preamble_ok, preamble_cut): {len(df)}")
assert len(df) == 158, f"Expected 158 clusterable rows, got {len(df)}"

# Save final country order
order_cols = ['Country', 'orig_cluster', 'status', 'extracted_words', 'short_flag']
df_order = df[order_cols].copy()
order_csv_path = os.path.join(data_dir, 'final_country_order.csv')
df_order.to_csv(order_csv_path, index=False)
print(f"Saved locked country order to: {order_csv_path}")

# Print country list in order
print("\n=== ORDERED COUNTRY LIST (158 COUNTRIES) ===")
for idx, c in enumerate(df['Country'], 1):
    print(f"{idx:3d}. {c}")


Total rows in ../data/processed/preamble_final.csv: 193
Clusterable rows (preamble_ok, preamble_cut): 158
Saved locked country order to: ../data/processed/final_country_order.csv

=== ORDERED COUNTRY LIST (158 COUNTRIES) ===
  1. Afghanistan
  2. Albania
  3. Algeria
  4. Andorra
  5. Angola
  6. Antigua and Barbuda
  7. Argentina
  8. Armenia
  9. Australia
 10. Azerbaijan
 11. Bahamas
 12. Bahrain
 13. Bangladesh
 14. Barbados
 15. Belarus
 16. Belize
 17. Benin
 18. Bhutan
 19. Bolivia
 20. Bosnia and Herz.
 21. Brazil
 22. Brunei
 23. Bulgaria
 24. Burkina Faso
 25. Burundi
 26. Cambodia
 27. Cameroon
 28. Canada
 29. Cabo Verde
 30. Central African Rep.
 31. Chad
 32. Chile
 33. China
 34. Colombia
 35. Comoros
 36. Congo
 37. Dem. Rep. Congo
 38. Costa Rica
 39. Côte d'Ivoire
 40. Croatia
 41. Cuba
 42. Czechia
 43. Djibouti
 44. Dominica
 45. Dominican Rep.
 46. Ecuador
 47. Egypt
 48. El Salvador
 49. Eq. Guinea
 50. Eritrea
 51. Estonia
 52. Ethiopia
 53. Fiji
 54. France
 55.

## 3. Cleaning Function from Notebook 01 Cell 44
Extract the cleaning function and custom stopword list verbatim from `notebooks/01_data_preprocessing.ipynb` cell 44, display its source code, and define it in the local environment.

In [3]:
with open('01_data_preprocessing.ipynb', 'r', encoding='utf-8') as f:
    nb01_json = json.load(f)

cell44_src = ''.join(nb01_json['cells'][44]['source'])
print("=== VERBATIM SOURCE CODE FROM NOTEBOOK 01 CELL 44 ===\n")
print(cell44_src)

# Define stopwords and tokenize_and_clean verbatim
stopwords = set(ENGLISH_STOP_WORDS) | {
    # Structural/legal words
    'shall', 'article', 'articles', 'art', 'section', 'subsection', 'provisions', 'law', 'laws', 
    'act', 'acts', 'office', 'member', 'members', 'parliament', 'assembly', 'council', 
    'federal', 'minister', 'ministers', 'authority', 'commission', 'court', 'courts', 'case', 
    'service', 'public', 'general', 'right', 'rights', 'offices', 
    'president', 'vice', 'chairman', 'secretary', 'person', 'persons', 'house', 'provided', 'functions',
    'paragraph', 'chapter', 'necessary', 'order', 'period', 'day', 'years', 'political', 'year', 'time',
    'people', 'committee', 'date', 'following', 'unless', 'matter', 'make', 'peoples', 'thirds', 'days',
    'manner', 'high', 'apply', 'matters', 'thirty', 'di', 'majlis', 'held', 'months',

    # Generic constitutional boilerplate
    'state', 'states', 'nation', 'national', 'country', 'citizens', 'citizen', 
    'government', 'preamble', 'hereby', 
    'thereof', 'therein', 'thereby', 'thereon', 'whereas', 'hence', 'thereafter',
    'may', 'upon', 'within', 'accordance', 'legal', 'ministry', 'terms', 'provision', 'amendment',
    'use',

    # Individual country names
    'china', 'australia', 'algerian', 'angolan', 'algeria', 'greek', 'turkish', 'angola', 'bhutan', 'nauru',
    'somalia', 'croatian', 'croatia', 'serbia', 'belarus', 'cambodia', 'having', 'vietnam', 'lao', 'tuvaluan',
    'tuvalu', 'papua', 'kosovo', 'pakistan', 'ukrainian', 'nepal', 'turkmenistan', 'russian', 'bangladesh',
    'turkey', 'armenia', 'azerbaijan', 'knesset', 'sierra', 'leone', 'botswana', 'riksdag',
    'malta', 'maldives', 'samoa', 'chinese', 'kim', 'portuguese', 'ii', 'sung', 'dprk', 'sri',
    'indonesia', 'lanka', 'india', 'south', 'guinea'
}


def tokenize_and_clean(text, stopwords=stopwords, min_token_len=2):
    if pd.isna(text):
        return [], ""
    text = str(text)

    # 1) Normalize case
    text = text.lower()

    # 2) Replace common 'article' patterns like "article 1", "art. 2" -> remove numeric part but keep 'article'
    text = re.sub(r'\bart(?:icle)?\.?\s*\d+\b', 'article', text)

    # 3) Remove standalone numbers (years, line numbers, numbered lists)
    text = re.sub(r'\b\d+\b', ' ', text)

    # 4) Remove common scrap/pagination markers like "page 5", "—page—", "©", etc.
    text = re.sub(r'page\s*\d+|\b©\b|—|–|_|[*#@]', ' ', text)

    # 5) Replace newlines and tabs with spaces
    text = text.replace('\n', ' ').replace('\r', ' ').replace('\t', ' ')

    # 6) Extract word tokens: allow letters and internal apostrophes (e.g., "people's")
    tokens = re.findall(r"[a-zA-Z']{2,}", text)

    # 7) Remove stopwords and tokens that are too short or purely punctuation
    tokens = [t for t in tokens if (t not in stopwords) and (len(t) >= min_token_len)]

    # 8) (Optional) You can remove tokens that are now just apostrophes or invalid:
    tokens = [t.strip("'") for t in tokens if t.strip("'")]

    # Re-join tokens to a cleaned string (useful if you want to vectorize later)
    cleaned_text = " ".join(tokens)

    return tokens, cleaned_text


=== VERBATIM SOURCE CODE FROM NOTEBOOK 01 CELL 44 ===

import re
import pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Load
df = pd.read_csv("../data/processed/constitution_dataset_cleaned.csv")

# Extend base stopwords — keep ideological words
stopwords = set(ENGLISH_STOP_WORDS) | {
    # Structural/legal words
    'shall', 'article', 'articles', 'art', 'section', 'subsection', 'provisions', 'law', 'laws', 
    'act', 'acts', 'office', 'member', 'members', 'parliament', 'assembly', 'council', 
    'federal', 'minister', 'ministers', 'authority', 'commission', 'court', 'courts', 'case', 
    'service', 'public', 'general', 'right', 'rights', 'offices', 
    'president', 'vice', 'chairman', 'secretary', 'person', 'persons', 'house', 'provided', 'functions',
    'paragraph', 'chapter', 'necessary', 'order', 'period', 'day', 'years', 'political', 'year', 'time',
    'people', 'committee', 'date', 'following', 'unless', 'matter', 'make', 'peoples', 'thirds',

## 4. Text Variants and Chunking
Generate Variant A ("raw": collapsed whitespace) and Variant B ("clean": verbatim cleaning function). Implement token-counting and chunking for texts exceeding 300 tokens:
- **Variant A**: Greedy sentence packing at sentence boundaries (`.?!`), breaking individual sentences > 300 tokens by tokens.
- **Variant B**: Consecutive windows of at most 300 tokens.

In [4]:
def count_tokens(text):
    return len(tokenizer.encode(str(text), add_special_tokens=False))

def chunk_variant_a(text, max_tokens=300):
    text = ' '.join(str(text).split())
    tot = count_tokens(text)
    if tot <= max_tokens:
        return [text], [tot]
    
    raw_sents = [s.strip() for s in re.split(r'(?<=[.?!])\s+', text) if s.strip()]
    units = []
    for s in raw_sents:
        s_toks = tokenizer.encode(s, add_special_tokens=False)
        if len(s_toks) <= max_tokens:
            units.append(s)
        else:
            for j in range(0, len(s_toks), max_tokens):
                sub_ids = s_toks[j:j+max_tokens]
                sub_t = tokenizer.decode(sub_ids, skip_special_tokens=True).strip()
                if sub_t:
                    units.append(sub_t)
                    
    chunks, curr = [], []
    for u in units:
        cand = ' '.join(curr + [u]) if curr else u
        if count_tokens(cand) <= max_tokens:
            curr.append(u)
        else:
            if curr:
                chunks.append(' '.join(curr))
            curr = [u]
    if curr:
        chunks.append(' '.join(curr))
        
    chunk_tokens = [count_tokens(c) for c in chunks]
    return chunks, chunk_tokens

def chunk_variant_b(clean_text, max_tokens=300):
    clean_text = ' '.join(str(clean_text).split())
    tot = count_tokens(clean_text)
    if tot <= max_tokens:
        return [clean_text], [tot]
    
    words = clean_text.split()
    chunks, curr = [], []
    for w in words:
        cand = ' '.join(curr + [w]) if curr else w
        if count_tokens(cand) <= max_tokens:
            curr.append(w)
        else:
            if curr:
                chunks.append(' '.join(curr))
            curr = [w]
    if curr:
        chunks.append(' '.join(curr))
        
    chunk_tokens = [count_tokens(c) for c in chunks]
    return chunks, chunk_tokens

# Apply text variants and chunking
df['raw_text'] = df['extracted_text'].apply(lambda x: ' '.join(str(x).split()))
clean_results = df['extracted_text'].apply(lambda x: tokenize_and_clean(x))
df['clean_tokens'] = clean_results.apply(lambda r: r[0])
df['clean_text'] = clean_results.apply(lambda r: r[1])

a_chunks, a_tokens = [], []
for t in df['raw_text']:
    c, tok = chunk_variant_a(t)
    a_chunks.append(c)
    a_tokens.append(tok)

b_chunks, b_tokens = [], []
for t in df['clean_text']:
    c, tok = chunk_variant_b(t)
    b_chunks.append(c)
    b_tokens.append(tok)

df['a_chunks'] = a_chunks
df['a_chunk_tokens'] = a_tokens
df['a_chunk_count'] = [len(c) for c in a_chunks]
df['a_total_tokens'] = [count_tokens(t) for t in df['raw_text']]

df['b_chunks'] = b_chunks
df['b_chunk_tokens'] = b_tokens
df['b_chunk_count'] = [len(c) for c in b_chunks]
df['b_total_tokens'] = [count_tokens(t) for t in df['clean_text']]

print(f"Processed chunking for {len(df)} countries.")
print(f"Variant A total chunks across all countries: {sum(df['a_chunk_count'])}")
print(f"Variant B total chunks across all countries: {sum(df['b_chunk_count'])}")


Token indices sequence length is longer than the specified maximum sequence length for this model (1487 > 384). Running this sequence through the model will result in indexing errors


Processed chunking for 158 countries.
Variant A total chunks across all countries: 276
Variant B total chunks across all countries: 177


## 5. Embedding Generation and Weighted Pooling
Embed all chunks, aggregate multi-chunk countries using token-weighted mean pooling, L2-normalise all embeddings to unit norm, and save to `data/processed/emb_final_raw.npy` and `data/processed/emb_final_clean.npy`. A second full embedding pass is executed to verify determinism.

In [5]:
def embed_chunks_weighted(chunk_texts_list, chunk_tokens_list):
    all_chunks = []
    chunk_counts = []
    for c_list in chunk_texts_list:
        all_chunks.extend(c_list)
        chunk_counts.append(len(c_list))
    
    all_embeddings = model.encode(all_chunks, batch_size=32, show_progress_bar=False, normalize_embeddings=False)
    
    country_embeddings = []
    idx = 0
    for i, count in enumerate(chunk_counts):
        c_embs = all_embeddings[idx:idx+count]
        c_weights = np.array(chunk_tokens_list[i], dtype=np.float32)
        idx += count
        
        weighted_mean = np.average(c_embs, axis=0, weights=c_weights)
        norm = np.linalg.norm(weighted_mean)
        l2_normed = weighted_mean / norm
        country_embeddings.append(l2_normed)
        
    return np.array(country_embeddings, dtype=np.float32)

print("Embedding Variant A (Raw) - Run 1...")
emb_final_raw = embed_chunks_weighted(df['a_chunks'], df['a_chunk_tokens'])
print("Embedding Variant B (Clean) - Run 1...")
emb_final_clean = embed_chunks_weighted(df['b_chunks'], df['b_chunk_tokens'])

print("Embedding Variant A (Raw) - Run 2 (Determinism Verification)...")
emb_final_raw_run2 = embed_chunks_weighted(df['a_chunks'], df['a_chunk_tokens'])
print("Embedding Variant B (Clean) - Run 2 (Determinism Verification)...")
emb_final_clean_run2 = embed_chunks_weighted(df['b_chunks'], df['b_chunk_tokens'])

# Save output embeddings
raw_npy_path = os.path.join(data_dir, 'emb_final_raw.npy')
clean_npy_path = os.path.join(data_dir, 'emb_final_clean.npy')
np.save(raw_npy_path, emb_final_raw)
np.save(clean_npy_path, emb_final_clean)
print(f"Saved raw embeddings to:  {raw_npy_path}")
print(f"Saved clean embeddings to: {clean_npy_path}")


Embedding Variant A (Raw) - Run 1...


Embedding Variant B (Clean) - Run 1...


Embedding Variant A (Raw) - Run 2 (Determinism Verification)...


Embedding Variant B (Clean) - Run 2 (Determinism Verification)...


Saved raw embeddings to:  ../data/processed/emb_final_raw.npy
Saved clean embeddings to: ../data/processed/emb_final_clean.npy


## 6. Checks 1 & 2: Structural Integrity and Determinism
Verify:
1. Shape is (158, 768) for both variants, no NaN or infinite values, and all row L2 norms are within $10^{-5}$ of 1.0.
2. Determinism across repeated model inference runs.

In [6]:
print("=== CHECK 1: SHAPE, NAN/INF, AND ROW NORMS ===\n")

for name, emb in [('Variant A (Raw)', emb_final_raw), ('Variant B (Clean)', emb_final_clean)]:
    print(f"[{name}]")
    print(f"  Shape:                 {emb.shape}")
    assert emb.shape == (158, 768), f"Unexpected shape for {name}: {emb.shape}"
    
    has_nan = np.isnan(emb).any()
    has_inf = np.isinf(emb).any()
    print(f"  Contains NaN:          {has_nan}")
    print(f"  Contains Inf:          {has_inf}")
    assert not has_nan, f"{name} contains NaN values!"
    assert not has_inf, f"{name} contains infinite values!"
    
    norms = np.linalg.norm(emb, axis=1)
    max_norm_err = np.max(np.abs(norms - 1.0))
    print(f"  Max |norm - 1.0|:      {max_norm_err:.2e}")
    assert np.allclose(norms, 1.0, atol=1e-5), f"{name} row norms violate 1e-5 tolerance!"
    print(f"  Assertion passed: all row norms within 1e-5 of 1.0\n")

print("=== CHECK 2: DETERMINISM ACROSS RUNS ===\n")

max_diff_raw = np.max(np.abs(emb_final_raw - emb_final_raw_run2))
max_diff_clean = np.max(np.abs(emb_final_clean - emb_final_clean_run2))

print(f"Max absolute difference (Variant A Run 1 vs Run 2): {max_diff_raw:.2e}")
print(f"Max absolute difference (Variant B Run 1 vs Run 2): {max_diff_clean:.2e}")
assert max_diff_raw < 1e-6, f"Variant A non-deterministic: max diff = {max_diff_raw}"
assert max_diff_clean < 1e-6, f"Variant B non-deterministic: max diff = {max_diff_clean}"
print("Assertion passed: zero divergence between embedding runs.")


=== CHECK 1: SHAPE, NAN/INF, AND ROW NORMS ===

[Variant A (Raw)]
  Shape:                 (158, 768)
  Contains NaN:          False
  Contains Inf:          False
  Max |norm - 1.0|:      1.19e-07
  Assertion passed: all row norms within 1e-5 of 1.0

[Variant B (Clean)]
  Shape:                 (158, 768)
  Contains NaN:          False
  Contains Inf:          False
  Max |norm - 1.0|:      1.19e-07
  Assertion passed: all row norms within 1e-5 of 1.0

=== CHECK 2: DETERMINISM ACROSS RUNS ===

Max absolute difference (Variant A Run 1 vs Run 2): 0.00e+00
Max absolute difference (Variant B Run 1 vs Run 2): 0.00e+00
Assertion passed: zero divergence between embedding runs.


## 7. Checks 3, 4, 5: Chunk Distributions, Short Preambles & Worked Example
Evaluate:
- Chunk count distribution, countries with $> 1$ chunk, token count statistics (mean, median, max), and top 10 countries with most chunks for each variant.
- Variant B countries with cleaned text under 20 tokens.
- Worked example for the country with the most chunks in Variant A (Iran: chunk token counts and first 80 characters of each chunk).

In [7]:
print("=== CHECK 3: CHUNK DISTRIBUTIONS AND TOKEN STATISTICS ===\n")

# Variant A
print("--- Variant A (Raw) ---")
print("Chunks per country distribution (value counts):")
print(df['a_chunk_count'].value_counts().sort_index())
a_multi = (df['a_chunk_count'] > 1).sum()
print(f"Countries with > 1 chunk: {a_multi}")
print(f"Token count mean:   {df['a_total_tokens'].mean():.2f}")
print(f"Token count median: {df['a_total_tokens'].median()}")
print(f"Token count max:    {df['a_total_tokens'].max()}")

print("\nTop 10 countries with most chunks (Variant A):")
top10_a = df.sort_values(by=['a_chunk_count', 'a_total_tokens'], ascending=[False, False]).head(10)
for _, r in top10_a.iterrows():
    print(f"  {r['Country']:<25} Tokens: {r['a_total_tokens']:4d}   Chunks: {r['a_chunk_count']:2d}")

# Variant B
print("\n--- Variant B (Clean) ---")
print("Chunks per country distribution (value counts):")
print(df['b_chunk_count'].value_counts().sort_index())
b_multi = (df['b_chunk_count'] > 1).sum()
print(f"Countries with > 1 chunk: {b_multi}")
print(f"Token count mean:   {df['b_total_tokens'].mean():.2f}")
print(f"Token count median: {df['b_total_tokens'].median()}")
print(f"Token count max:    {df['b_total_tokens'].max()}")

print("\nTop 10 countries with most chunks (Variant B):")
top10_b = df.sort_values(by=['b_chunk_count', 'b_total_tokens'], ascending=[False, False]).head(10)
for _, r in top10_b.iterrows():
    print(f"  {r['Country']:<25} Tokens: {r['b_total_tokens']:4d}   Chunks: {r['b_chunk_count']:2d}")

print("\n=== CHECK 4: VARIANT B CLEANED TEXT UNDER 20 TOKENS ===\n")
under_20_b = df[df['b_total_tokens'] < 20][['Country', 'b_total_tokens']]
print(f"Number of countries under 20 tokens: {len(under_20_b)}")
for _, r in under_20_b.iterrows():
    print(f"  {r['Country']}: {r['b_total_tokens']} tokens")

print("\n=== CHECK 5: WORKED EXAMPLE (COUNTRY WITH MOST CHUNKS IN VARIANT A) ===\n")
max_chunks_country = top10_a.iloc[0]['Country']
max_c_row = df[df['Country'] == max_chunks_country].iloc[0]
print(f"Country: {max_chunks_country} (Total Chunks: {max_c_row['a_chunk_count']}, Total Tokens: {max_c_row['a_total_tokens']})")
for k, (ch, tk) in enumerate(zip(max_c_row['a_chunks'], max_c_row['a_chunk_tokens']), 1):
    print(f"  Chunk {k:2d} ({tk:3d} tokens): {repr(ch[:80])}")


=== CHECK 3: CHUNK DISTRIBUTIONS AND TOKEN STATISTICS ===

--- Variant A (Raw) ---
Chunks per country distribution (value counts):
a_chunk_count
1     95
2     40
3     14
4      4
5      1
6      2
11     1
15     1
Name: count, dtype: int64
Countries with > 1 chunk: 63
Token count mean:   357.39
Token count median: 254.0
Token count max:    3934

Top 10 countries with most chunks (Variant A):
  Iran                      Tokens: 3934   Chunks: 15
  Papua New Guinea          Tokens: 2344   Chunks: 11
  Algeria                   Tokens: 1487   Chunks:  6
  Egypt                     Tokens: 1477   Chunks:  6
  China                     Tokens: 1402   Chunks:  5
  Cameroon                  Tokens:  973   Chunks:  4
  Thailand                  Tokens:  924   Chunks:  4
  Morocco                   Tokens:  744   Chunks:  4
  Iraq                      Tokens:  707   Chunks:  4
  Bahrain                   Tokens:  867   Chunks:  3

--- Variant B (Clean) ---
Chunks per country distribution (va

## 8. Checks 6 & 7: Cosine Similarities and Length Bias Diagnostics
Evaluate:
- Mean pairwise cosine similarity across countries for both variants (distinct pairs and all pairs).
- Spearman rank correlation between document length (`extracted_words`) and cosine similarity to the mean embedding vector for both variants.

In [8]:
print("=== CHECK 6: MEAN PAIRWISE COSINE SIMILARITY ===\n")

for name, emb in [('Variant A (Raw)', emb_final_raw), ('Variant B (Clean)', emb_final_clean)]:
    cos_sim_matrix = np.dot(emb, emb.T)
    n = emb.shape[0]
    off_diag_mask = ~np.eye(n, dtype=bool)
    mean_off_diag = cos_sim_matrix[off_diag_mask].mean()
    mean_all = cos_sim_matrix.mean()
    print(f"[{name}]")
    print(f"  Mean pairwise cosine similarity (distinct pairs, n={n*(n-1)}): {mean_off_diag:.6f}")
    print(f"  Mean all-pairs cosine similarity (including diagonal, n={n*n}):  {mean_all:.6f}\n")

print("=== CHECK 7: SPEARMAN CORRELATION WITH EXTRACTED_WORDS ===\n")

words = df['extracted_words'].values

for name, emb in [('Variant A (Raw)', emb_final_raw), ('Variant B (Clean)', emb_final_clean)]:
    mean_vector = np.mean(emb, axis=0)
    mean_vector_norm = mean_vector / np.linalg.norm(mean_vector)
    sim_to_mean = np.dot(emb, mean_vector_norm)
    
    rho, pval = spearmanr(words, sim_to_mean)
    print(f"[{name}]")
    print(f"  Spearman rho: {rho:.6f}")
    print(f"  p-value:      {pval:.4e}\n")


=== CHECK 6: MEAN PAIRWISE COSINE SIMILARITY ===

[Variant A (Raw)]
  Mean pairwise cosine similarity (distinct pairs, n=24806): 0.507854
  Mean all-pairs cosine similarity (including diagonal, n=24964):  0.510969

[Variant B (Clean)]
  Mean pairwise cosine similarity (distinct pairs, n=24806): 0.496637
  Mean all-pairs cosine similarity (including diagonal, n=24964):  0.499823

=== CHECK 7: SPEARMAN CORRELATION WITH EXTRACTED_WORDS ===

[Variant A (Raw)]
  Spearman rho: 0.056751
  p-value:      4.7879e-01

[Variant B (Clean)]
  Spearman rho: 0.260793
  p-value:      9.3447e-04

